# Trajectory Follow Comparison
Plots of the closed-loop simulations saved by `trajectory_generation.ipynb` (no simulation here).

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os

import numpy as np
import pandas as pd

from src.sim.simulation_io import (experiment_path, experiment_title, load_simulations,
                                   load_reference, plot_angle, plot_torque, plot_activations)
from src.sim.trajectory import phase_spans
from src.utils.plotting import plot_stacked

## Import Data

In [ ]:
TRAJ_NAME = "minjerk_500ms"  # "step"

W_REF = 0.5
INIT_ANGLE = 0    # [°]
ANGLE_REF = 130   # [°]

K_P = 1.0
K_I = 1.0
K_D = 0.0

DELAY_MS = 150.0   # [ms], 0 = no delay
FEEDFORWARD = True # False for a step

MODELS = None      # None = all the simulations of the folder, or e.g. ["LUT", "solver", ...]

FOLDER = experiment_path(TRAJ_NAME, INIT_ANGLE, ANGLE_REF, W_REF, K_P, K_I, K_D,
                         delay_ms=DELAY_MS, ff=FEEDFORWARD)
if not os.path.isdir(FOLDER):
    parent = os.path.dirname(FOLDER)
    available = sorted(os.listdir(parent)) if os.path.isdir(parent) else []
    raise FileNotFoundError(f"{FOLDER} not found.\nIn {parent}: {available}")

sims, parameters = load_simulations(FOLDER, labels=MODELS)
angle_ref = load_reference(FOLDER)

print("Folder:", FOLDER)
print("Loaded:", list(sims))
pd.json_normalize({k: v for k, v in parameters.items() if k != "allocators"}).T

## Summary

In [ ]:
direction = np.sign(ANGLE_REF - INIT_ANGLE) or 1.0

rows = {}
for name, df in sims.items():
    valid = df.dropna(subset=["theta"])
    e = valid.theta_d - valid.theta
    row = {"final angle [deg]": valid.theta.iloc[-1],
           "final error [deg]": e.iloc[-1],
           "overshoot [deg]": max(0.0, (direction * (valid.theta - ANGLE_REF)).max()),
           "RMS error [deg]": np.sqrt((e ** 2).mean()),
           "final tau_req [Nm]": valid.tau_req.iloc[-1],
           "stopped at [ms]": valid.t.iloc[-1] if len(valid) < len(df) else np.nan}
    if "phase" in valid:
        moving = valid.phase.isin(["move", "locked"])
        row["RMS error move+locked [deg]"] = np.sqrt((e[moving] ** 2).mean())
    rows[name] = row

summary = pd.DataFrame(rows).T
summary.round(3)

## Desired Angle

In [ ]:
t = angle_ref["t"].to_numpy()
spans = phase_spans(t, angle_ref["phase"].to_numpy(), t[1] - t[0]) if "phase" in angle_ref else None
plot_stacked(title=f"Desired angle ({INIT_ANGLE}° → {ANGLE_REF}°)", x=t,
             y=[angle_ref["theta_d"], angle_ref["theta_d_dot"], angle_ref["theta_d_ddot"]],
             y_labels=[r"$\theta_d$ [deg]", r"$\dot{\theta}_d$ [deg/s]", r"$\ddot{\theta}_d$ [deg/s$^2$]"],
             x_label="Time [ms]", spans=spans)

## Plots

In [ ]:
PLOT_T_MIN = None   # [ms], None = from the start
PLOT_T_MAX = None   # [ms], None = to the end (e.g. 1000 to zoom on the movement)
SHOW_PHASES = True
SAVE_PLOTS = False

title = experiment_title(TRAJ_NAME, INIT_ANGLE, ANGLE_REF, W_REF, K_P, K_I, K_D,
                         delay_ms=DELAY_MS, ff=FEEDFORWARD)
save_dir = FOLDER.replace("simulations", "plots", 1) if SAVE_PLOTS else None
window = dict(t_min=PLOT_T_MIN, t_max=PLOT_T_MAX)

plot_angle(sims, title=f"Elbow angle ({title})", show_phases=SHOW_PHASES, save_dir=save_dir, **window)
plot_torque(sims, title=f"Torque ({title})", save_dir=save_dir, **window)
plot_activations(sims, title=f"Activations ({title})", save_dir=save_dir, **window)
if all("act_0" in df for df in sims.values()):
    plot_activations(sims, title=f"Activation states ({title})", state=True,
                     save_dir=save_dir, **window)